# Group Assignment Attribution

**Attribution:** This notebook was developed as part of a group assignment. It is included for portfolio transparency and does not claim sole authorship.

# *Title and Objective*

## Economic Spelling Correction (NLP Interface)
This notebook builds an NLP-based spelling correction system using domain-specific vocabulary. The system leverages a corpus from *The Principles of Economics, with Applications to Practical Problems* by Frank A. Fetter, applying edit-distance, bigram probabilities, and word frequency to detect and suggest corrections for misspelled or contextually confusing words. An interactive user interface is developed using Gradio for real-time feedback.

# 📌 Step 0: Install & Import Libraries

## Step 0: Imports & Dependencies

In [ ]:
# ========== Step 0: Imports & Dependencies (auto-install if missing) ==========
try:
    import gradio as gr
    import requests
    import re, math, html
    from collections import Counter, defaultdict
    from wordfreq import top_n_list, zipf_frequency
except Exception:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "gradio", "wordfreq", "requests"])
    import gradio as gr
    import requests
    import re, math, html
    from collections import Counter, defaultdict
    from wordfreq import top_n_list, zipf_frequency

# 🧹 Step 1: Preprocessing — Corpus Initialization

This step loads the Economics corpus from Frank A. Fetter's *The Principles of Economics, with Applications to Practical Problems* via Project Gutenberg.
It converts the text to lowercase, tokenizes it into words, builds the vocabulary, and calculates unigram and bigram frequencies. The processed data is used later for spelling correction and contextual suggestions.


In [ ]:
# ========== Global Variables ==========
SYSTEM_INITIALIZED = False
FREQ = None
VOCAB = None
BG = None
V = 0
TOTAL = 0

# ========== Step 1: System Initialization ==========
def initialize_system():
    """Download corpus, build vocabulary (corpus + wordfreq), unigrams and bigrams."""
    global SYSTEM_INITIALIZED, FREQ, VOCAB, BG, V, TOTAL
    if SYSTEM_INITIALIZED:
        return "✅ System already initialized!"

    try:
        CORPUS_URL = "https://www.gutenberg.org/ebooks/40077.txt.utf-8"
        text = requests.get(CORPUS_URL, timeout=30).text.lower()

        # Tokenize corpus
        tokens = re.findall(r"[a-z]+", text)

        # Unigrams / vocab
        FREQ = Counter(tokens)
        VOCAB = set(tokens)
        VOCAB |= set(top_n_list('en', n=100_000))  # add common english

        # Bigrams
        BG = defaultdict(int)
        for a, b in zip(tokens, tokens[1:]):
            BG[(a, b)] += 1

        V = max(1, len(VOCAB))
        TOTAL = sum(FREQ.values())
        SYSTEM_INITIALIZED = True

        return (
            "✅ System initialized!\n"
            "📘 Corpus: The Principles of Economics, with Applications to Practical Problems (Frank A. Fetter)\n"
            f"📊 Vocabulary: {len(VOCAB):,}\n"
            f"📚 Tokens: {TOTAL:,}\n"
            f"🔗 Bigrams: {len(BG):,}"
        )
    except Exception as e:
        return f"❌ Initialization failed: {str(e)}"

def reset_system():
    """Reset state so you can re-initialize cleanly."""
    global SYSTEM_INITIALIZED, FREQ, VOCAB, BG, V, TOTAL
    SYSTEM_INITIALIZED = False
    FREQ = None
    VOCAB = None
    BG = None
    V = 0
    TOTAL = 0
    return "🔄 System reset. Click Initialize again."

# 📊 Step 2: Exploratory Data Analysis (EDA)

We analyze the processed corpus to understand vocabulary patterns, frequency distribution, and richness.
Visual tools such as bar charts and word clouds provide insight into the most commonly used economic terms.


In [ ]:
# ========== Step 2: Simple analysis & wordcloud ==========
# Ensure system is initialized before proceeding with EDA
if not SYSTEM_INITIALIZED:
    print("Attempting to initialize system...")
    init_status = initialize_system()
    print(init_status)
    if not SYSTEM_INITIALIZED:
        print("Initialization failed. Cannot perform EDA.")
    else:
        print("Initialization successful. Proceeding with EDA.")

# Import necessary libraries for EDA
import matplotlib.pyplot as plt
from wordcloud import WordCloud

if SYSTEM_INITIALIZED:
    print("\n📈 Basic stats")
    print(f"Total words: {TOTAL:,}")
    print(f"Unique words: {len(VOCAB):,}")
    if TOTAL:
        print(f"Vocab richness: {len(VOCAB)/TOTAL:.2%}")
    else:
        print("Cannot calculate richness: Total tokens is zero.")

    top20 = dict(FREQ.most_common(20))

    plt.figure(figsize=(14,5))
    plt.bar(range(len(top20)), list(top20.values()))
    plt.xticks(range(len(top20)), list(top20.keys()), rotation=45, ha='right')
    plt.title("Top 20 Most Frequent Terms")
    plt.tight_layout()
    plt.show()

    try:
        wc_text = " ".join([w for w, _ in FREQ.most_common(200)])
        if wc_text:
            wc = WordCloud(width=900, height=400, background_color="white").generate(wc_text)
            plt.figure(figsize=(12,5))
            plt.imshow(wc, interpolation="bilinear")
            plt.axis("off")
            plt.title("Word Cloud")
            plt.show()
        else:
            print("Not enough data to generate WordCloud from top 200 words.")
    except Exception as e:
        print("WordCloud error:", e)
else:
    print("System not initialized. Skipping EDA.")

In [ ]:
# 📖 Display First 250 Words and Sentences with Tokens from the Economics Corpus
import requests
import re
import textwrap  # for neat paragraph wrapping

CORPUS_URL = "https://www.gutenberg.org/ebooks/40077.txt.utf-8"

try:
    # Download corpus
    text = requests.get(CORPUS_URL, timeout=30).text.lower()
    tokens = re.findall(r"[a-z]+", text)

    print(f"📘 Total Words in Corpus: {len(tokens):,}")

    # --- First 250 words as paragraph ---
    print("📝 First 250 Words (Paragraph View):\n")
    first_250 = " ".join(tokens[:250])
    print(textwrap.fill(first_250, width=150))  # wrap ~150 chars per line

    # --- Sentence tokenization for first 250 words ---
    text_excerpt = first_250
    sentences = re.split(r'(?<=[.!?]) +', text_excerpt)  # split by punctuation + space
    sentences = [s.strip() for s in sentences if s.strip()]

    print("\n🗨️ Sentences with Tokens:\n")
    for i, sentence in enumerate(sentences, 1):
        words_in_sentence = re.findall(r"[a-z]+", sentence.lower())
        print(f"Sentence {i}: {sentence}")
        print(f"Tokens: {words_in_sentence}\n")

except Exception as e:
    print("❌ Failed to download or process the corpus:", e)


# 🔹 Step 3: Bigram Log Probability

We calculate bigram log-probabilities with add-one smoothing. This helps the system evaluate whether a candidate correction fits in its sentence context.

In [ ]:
# ========== Step 3: Bigram Log Probability ==========
def logP_bigram(w1, w2, alpha=1.0):
    """Add-one smoothed bigram log-probability."""
    if not SYSTEM_INITIALIZED:
        return 0.0
    if not w1 or not w2:
        num = FREQ.get(w2, 0) + alpha
        den = TOTAL + alpha * V
        return math.log(num / den)
    num = BG.get((w1, w2), 0) + alpha
    den = FREQ.get(w1, 0) + alpha * V
    return math.log(num / den)

# 🔹 Step 4: Edit Distance Functions

We implement single-edit and double-edit candidate generation (insertions, deletions, substitutions, transpositions).  These are used to propose spelling correction candidates.


In [ ]:
# ========== Step 4: Edit-Distance Helpers ==========
letters = "abcdefghijklmnopqrstuvwxyz"

def edits1(w):
    splits = [(w[:i], w[i:]) for i in range(len(w)+1)]
    deletes = [L+R[1:] for L,R in splits if R]
    transposes = [L+R[1]+R[0]+R[2:] for L,R in splits if len(R)>1]
    replaces = [L+c+R[1:] for L,R in splits if R for c in letters]
    inserts = [L+c+R for L,R in splits for c in letters]
    return set(deletes + transposes + replaces + inserts)

def edits2(w, cap=4000):
    out = set()
    for e1 in edits1(w):
        for e2 in edits1(e1):
            if VOCAB and e2 in VOCAB:
                out.add(e2)
                if len(out) >= cap:
                    return out
    return out

def lev(a, b):
    if a == b: return 0
    if not a: return len(b)
    if not b: return len(a)
    prev = list(range(len(b)+1))
    for i, ca in enumerate(a, 1):
        curr = [i]
        for j, cb in enumerate(b, 1):
            curr.append(min(prev[j]+1, curr[j-1]+1, prev[j-1]+(ca != cb)))
        prev = curr
    return prev[-1]


# Top-3 Candidate Suggestions

In [ ]:
# ========== Step 5: Top-3 Candidate Suggestions (Engine from Code 1) ==========
def suggest_top3(word, want=3):
    """Rank by (lev, start/end-letter match, -freq, -zipf, alpha) with sensible fallbacks."""
    if not SYSTEM_INITIALIZED or not VOCAB:
        return []

    w = word.lower()
    if w in VOCAB:
        return []

    # 1-edit candidates + expand if needed
    cand = {c for c in edits1(w) if c in VOCAB}
    if len(cand) < want:
        cand.update(edits2(w, cap=2000))
    cand = {c for c in cand if abs(len(c) - len(w)) <= 3}
    if len(cand) < want:
        extra = edits2(w, cap=3000)
        cand.update(extra)
        cand = {c for c in cand if abs(len(c) - len(w)) <= 4}

    def score(c):
        return (
            lev(w, c),
            0 if c.startswith(w[0]) else 1,
            0 if c.endswith(w[-1]) else 1,
            -FREQ.get(c, 0),
            -zipf_frequency(c, 'en'),
            c
        )

    sorted_cands = sorted(cand, key=score)

    # Prefer a plausible "correct" first if close enough
    correct = None
    for c in sorted_cands:
        if lev(w, c) <= 2 and FREQ.get(c, 0) > 0:
            correct = c
            break

    final_list = []
    if correct:
        final_list.append(correct)

    for c in sorted_cands:
        if c != correct and c not in final_list:
            final_list.append(c)
        if len(final_list) >= want:
            break

    return final_list[:want]

# Test

print(suggest_top3("econmics"))
print(suggest_top3("princples"))
print(suggest_top3("gvernement"))



# 🔹Step 6: Spell Checker (with real-word context)

We implement a rule-based and context-aware spell checker that detects both non-word errors and real-word misuses. Non-words are corrected using candidate generation and ranking from previous steps. For real-words, the system uses bigram probabilities to evaluate if a nearby alternative would better fit the sentence context. A set of common confusion word pairs and reliable stopwords are used to refine suggestions.

In [ ]:
def _fmt_line(orig: str, label: str, cands: list[str]) -> str:
    """Standard row: '❌ Word (label) → cand1, cand2, cand3'"""
    cand_str = ", ".join(cands[:3]) if cands else "—"
    return f"❌ {orig} ({label}) → {cand_str}"


In [ ]:
# ========== Step 6: Spell Checker (with real-word context) ==========
COMMON_CONFUSIONS = {
    "accept": "except", "except": "accept", "advise": "advice",
    "affect": "effect", "effect": "affect", "insure": "ensure", "ensure": "insure"
}

RELIABLE = {
    'the','a','an','and','in','on','at','to','for','of','with','by','this','that',
    'is','are','was','were','be','been','being','have','has','had','do','does','did',
    'will','would','can','may','might','not','no','yes','all','any','some','many','more','most'
}

def check_spelling(text):
    """Non-words -> suggestions; real-words -> context bigram check for gentle hints."""
    if not SYSTEM_INITIALIZED:
        return "❌ Please initialize system first."
    if not text.strip():
        return "📝 Enter text to check."

    words = re.findall(r"[A-Za-z']+", text)[:5000]
    lower = [w.lower() for w in words]
    lines = []

    for i, w in enumerate(lower):
        # Non-word
        if w not in VOCAB:
            sugs = suggest_top3(w)
            lines.append(f"❌ {words[i]} (non-word) → {', '.join(sugs) if sugs else 'no suggestions'}")
            continue

        if w in COMMON_CONFUSIONS:
            lines.append(f"⚠️ {words[i]} (real-word) → maybe '{COMMON_CONFUSIONS[w]}'?")
            continue

        # Real-word checks
        if w in RELIABLE:
            continue

        prev = lower[i-1] if i > 0 else None
        nxt  = lower[i+1] if i+1 < len(lower) else None
        base = (logP_bigram(prev, w) if prev else 0) + (logP_bigram(w, nxt) if nxt else 0)

        cands = suggest_top3(w) or list({c for c in edits1(w) if c in VOCAB})[:3]
        best, best_gain = None, 0
        for c in cands:
            score = (logP_bigram(prev, c) if prev else 0) + (logP_bigram(c, nxt) if nxt else 0)
            gain = score - base
            if gain > best_gain:
                best_gain = gain
                best = c

        if w in COMMON_CONFUSIONS:
            lines.append(f"⚠️ {words[i]} (real-word) → maybe '{COMMON_CONFUSIONS[w]}'?")
            continue

        if best and best_gain > math.log(1.05) and lev(w, best) <= 2 and best != w:
            lines.append(f"⚠️ {words[i]} (real-word) → maybe '{best}'?")

    return "✅ No errors detected!" if not lines else "🔎 Results:\n\n" + "\n".join(lines)

In [ ]:
def split_check(text):
    report = check_spelling(text)
    if report.startswith("✅"):
        return "No non-word errors.", "No real-word errors."

    non_word_lines = []
    real_word_lines = []
    for line in report.splitlines():
        if line.startswith("❌"):   # non-word
            non_word_lines.append(line)
        elif line.startswith("⚠️"): # real-word confusion
            real_word_lines.append(line)

    return "\n".join(non_word_lines) or "No non-word errors.", \
           "\n".join(real_word_lines) or "No real-word errors."

In [ ]:
# ========== Test Spelling Correction System ==========

TEST_SENTENCES = [
    "I will except the gift.",   # should be "accept"
    "The affect of the drug was strong.", # should be "effect"
    "Thiss is an exmple.",        # non-words
]

def run_tests():
    print("===== Spelling Correction Tests =====\n")
    for i, sent in enumerate(TEST_SENTENCES, 1):
        print(f"Test {i}: {sent}")
        result = check_spelling(sent)
        print(result, "\n")

# Run the tests
run_tests()

# 🔧 Step 6: Other Tools

This section includes supporting tools that enrich the functionality of the spelling correction system, such as vocabulary exploration, dictionary lookup, and live spelling preview.


In [ ]:
# ========== Step 7: Other Tools ==========
def clear_text():
    return ""

def search_vocabulary(query=""):
    if not SYSTEM_INITIALIZED or not VOCAB:
        return "❌ Please initialize first."
    if not query.strip():
        most_common = FREQ.most_common(20)
        return "\n".join([f"• {w} ({c})" for w, c in most_common])
    query = query.lower()
    matches = [w for w in VOCAB if query in w][:50]
    if not matches:
        return f"❌ No words found with '{query}'"
    return "\n".join([f"• {w} ({FREQ.get(w, 0)} occurrences)" for w in sorted(matches)[:20]])

def analyze_word(word):
    if not SYSTEM_INITIALIZED or not VOCAB:
        return "❌ Please initialize first."
    if not word.strip():
        return "📝 Enter word to analyze."
    w = word.lower().strip()
    result = f"🔍 Analysis for '{word}':\n\n"
    result += f"📝 Word: {w}\n📏 Length: {len(w)}\n"
    in_vocab = w in VOCAB
    result += f"📚 In vocab: {'✅ Yes' if in_vocab else '❌ No'}\n"
    if in_vocab:
        result += f"📊 Corpus frequency: {FREQ.get(w,0)}\n"
        result += f"🌍 Zipf freq: {zipf_frequency(w, 'en'):.2f}\n"
    else:
        sugs = suggest_top3(w)
        if sugs:
            result += "💡 Suggestions:\n" + "\n".join([f"• {s}" for s in sugs])
    return result

## 📚 7a) Dictionary Lookup Tool
- Accepts a word or phrase.
- Queries: dictionaryapi.dev → Wikipedia → Wiktionary.
- Shows definitions and explanations.
- Provides fallback definitions for individual words if a phrase fails.

---

In [ ]:
# ==============================================
# Step 7: Dictionary & Vocabulary Tools
## # --- 7a) Dictionary
# ==============================================
import requests
import re

# Utility: Format output
def _fmt(title: str, defs, icon="📖"):
    if isinstance(defs, str):
        return f"{icon} {title}:\n{defs}"
    if not defs:
        return f"❌ No definitions found for '{title}'."
    return f"{icon} {title}:\n" + "\n".join(defs[:6])

# 🔍 Main Dictionary Lookup Function
def dictionary_lookup(text: str, show_single_words_when_phrase_found: bool = False):
    text = (text or "").strip()
    if not text:
        return "📝 Enter a word or a phrase."

    tokens = re.findall(r"[A-Za-z']+", text)[:5]
    if not tokens:
        return "⚠️ Please type a valid word or phrase."
    phrase = " ".join(tokens).lower()

    outputs = []
    phrase_found = False

    # 1️⃣ Try multiple phrase formats in Wikipedia first
    for variant in [phrase, phrase.replace(" ", "_"), phrase.replace(" ", "-"), phrase.replace(" ", "")]:
        wiki = _wiki_summary_any(variant)
        if wiki:
            outputs.append(_fmt(variant.title(), wiki, icon="📚"))
            phrase_found = True
            break

    # 2️⃣ Try Wiktionary
    if not phrase_found:
        wikt = _wiktionary_defs_any(phrase)
        if wikt:
            outputs.append(_fmt(phrase, wikt, icon="📖"))
            phrase_found = True

    # 3️⃣ Try dictionaryapi.dev
    if not phrase_found:
        for cand in [phrase, phrase.replace(" ", "-"), phrase.replace(" ", ""), phrase.replace(" ", "_")]:
            defs = _defs_dictionaryapi(cand)
            if defs:
                outputs.append(_fmt(cand, defs, icon="📖"))
                phrase_found = True
                break

    # 4️⃣ Optional: Corpus-based economic phrase dictionary (if available)
    if not phrase_found and 'phrase_dict' in globals():
        local_def = lookup_phrase(phrase, phrase_dict)  # from your local corpus-based dictionary
        if local_def and "not found" not in local_def.lower():
            outputs.append(_fmt(phrase + " (Corpus)", local_def, icon="📘"))
            phrase_found = True

    # 5️⃣ Optional: Fallback to single words
    if not phrase_found or show_single_words_when_phrase_found:
        if outputs and not show_single_words_when_phrase_found:
            outputs.append("--- Single word definitions ---")
        for w in tokens:
            defs = _defs_dictionaryapi(w.lower())
            if not defs:
                defs = _wiktionary_defs_any(w.lower())
            outputs.append(_fmt(w.lower(), defs))

    return "\n\n".join(outputs) if outputs else "❌ No definitions found."


# ✅ DictionaryAPI.dev
def _defs_dictionaryapi(term: str):
    try:
        safe = requests.utils.quote(term)
        url = f"https://api.dictionaryapi.dev/api/v2/entries/en/{safe}"
        resp = requests.get(url, timeout=12).json()
        if not (isinstance(resp, list) and resp):
            return []
        meanings = resp[0].get("meanings", [])
        defs = []
        for m in meanings:
            part = m.get("partOfSpeech", "")
            for d in m.get("definitions", []):
                defi = (d.get("definition") or "").strip()
                if defi:
                    defs.append(f"({part}) {defi}")
        return defs
    except Exception:
        return []

# ✅ Wikipedia Summary
def _wiki_summary_any(term: str):
    def _rest_summary(title: str):
        try:
            safe = requests.utils.quote(title.replace(" ", "_"))
            url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{safe}"
            js = requests.get(url, timeout=12, headers={"accept": "application/json"}).json()
            if js and js.get("extract") and js.get("type") != "disambiguation":
                return js["extract"].strip()
        except Exception:
            pass
        return ""
    s = _rest_summary(term)
    if s:
        return s
    try:
        q = requests.utils.quote(term)
        url = f"https://en.wikipedia.org/w/api.php?action=query&list=search&srsearch={q}&srlimit=1&utf8=&format=json"
        js = requests.get(url, timeout=12).json()
        hits = js.get("query", {}).get("search", [])
        if hits:
            title = hits[0].get("title", "")
            return _rest_summary(title)
    except Exception:
        pass
    return ""

# ✅ Wiktionary API
def _wiktionary_defs_any(term: str):
    import requests, re
    try:
        url = f"https://en.wiktionary.org/api/rest_v1/page/definition/{requests.utils.quote(term)}"
        headers = {"User-Agent": "Mozilla/5.0"}
        js = requests.get(url, headers=headers, timeout=10).json()
        entries = js.get("en")
        defs = []
        if entries:
            for entry in entries:
                part = entry.get("partOfSpeech", "")
                for d in entry.get("definitions", []):
                    raw_def = d.get("definition", "").strip()
                    clean_def = re.sub(r'<.*?>', '', raw_def)  # REMOVE HTML TAGS
                    if clean_def:
                        defs.append(f"({part}) {clean_def}")
        return defs
    except:
        return []

    d = _defs_exact(term)
    if d:
        return d
    try:
        q = requests.utils.quote(term)
        url = f"https://en.wiktionary.org/w/api.php?action=query&list=search&srsearch={q}&srlimit=1&utf8=&format=json"
        js = requests.get(url, timeout=12).json()
        hits = js.get("query", {}).get("search", [])
        if hits:
            title = hits[0].get("title", "")
            return _defs_exact(title)
    except Exception:
        pass
    return []

## 🪄 7b) Auto-Correct UI Helper
- Fixes non-word errors using top-3 candidate suggestions.
- Keeps original casing and punctuation.
- Enhances user input in Gradio interface.

---

In [ ]:
# --- 7b) Auto-Correct (UI helper; uses Code-1 engine for non-words only) ---
def ui_autocorrect(text: str):
    if not SYSTEM_INITIALIZED or not text:
        return text or ""
    def repl(m):
        token = m.group(0)
        lw = token.lower()
        if lw in VOCAB:
            return token
        sugs = suggest_top3(lw)
        return sugs[0] if sugs else token
    # Only touch alphabetic tokens; keep punctuation/casing as-is
    return re.sub(r"[A-Za-z]+", repl, text)

## 🔴 7c) Live Wavy Underline Preview
- HTML preview that highlights unrecognized words.
- Uses a red wavy underline for misspellings.
- Improves UX by offering visual feedback on typos without modifying the text.

In [ ]:
# --- 7c) Live Wavy Underline Preview (non-invasive) ---
_WAVY_CSS = """
<style>
.wavy-underline { text-decoration: underline wavy red; text-decoration-thickness: 1.5px; }
.editor-shell {
  background:#ffffff; color:#000000;
  border:1px solid #e5e7eb; border-radius:8px;
  padding:12px; min-height:180px; white-space:pre-wrap; line-height:1.6;
}
.dim { opacity:.6 }
</style>
"""
def highlight_html(text: str):
    if not text:
        return _WAVY_CSS + '<div class="editor-shell dim">Enter text…</div>'
    # Mark tokens not in VOCAB as misspelled (simple live preview)
    bad = set()
    for w in re.findall(r"[A-Za-z']+", text):
        if w.lower() not in (VOCAB or set()):
            bad.add(w)
    def paint(m):
        t = m.group(0)
        safe = html.escape(t)
        return f'<span class="wavy-underline">{safe}</span>' if t in bad else safe
    body = re.sub(r"[A-Za-z']+", paint, html.escape(text))
    return _WAVY_CSS + f'<div id="live-editor" class="editor-shell">{body}</div>'

# 🛠️ Step 8: Gradio Interface

We developed an interactive Gradio interface to provide a user-friendly environment for testing the spelling correction system. The interface includes live preview with inline highlighting, spell-check and auto-correction buttons, dictionary lookup, vocabulary search, and word analysis. It allows users to input raw text and receive real-time suggestions and corrections, enhancing usability for non-technical users.

In [ ]:
# ========== Step 8: Gradio Interface ==========
def create_interface():
    with gr.Blocks(title="Spelling Correction — Economics Edition") as demo:
        gr.HTML(
            "<h2 style='text-align:center;'>📘 Advanced NLP Spelling Correction System<br>"
            "Corpus: <em>The Principles of Economics, with Applications to Practical Problem</em> by Frank A. Fetter</h2>"
            "<p style='text-align:center;opacity:.8;'>Inline wavy underlines for misspellings & confusable words, live as you type.</p>"
        )

        # Top control buttons
        with gr.Row():
            init_btn  = gr.Button("🚀 Initialize System", variant="primary")
            reset_btn = gr.Button("🔄 Reset System")
        init_output = gr.Textbox(label="Status", interactive=False)

        with gr.Row():
            with gr.Column(scale=6):
                gr.Markdown("### 📥 Input Text")
                text_input = gr.Textbox(placeholder="Enter or paste text here...", lines=10)
                live_preview = gr.HTML(value=highlight_html(""), label="Live Preview")

                with gr.Row():
                    check_btn = gr.Button("✅ Check Spelling")
                    auto_btn  = gr.Button("✨ Auto-Correct")
                    clear_btn = gr.Button("🗑️ Clear")

                with gr.Column():
                  with gr.Column():
                     gr.Markdown("### ❌ Non-Word Errors")
                     nonword_output = gr.Textbox(lines=6, interactive=False, label="Non-Word Errors")

                  with gr.Column():
                    gr.Markdown("### ⚠️ Real-Word Errors")
                    realword_output = gr.Textbox(lines=6, interactive=False, label="Real-Word Errors")


            with gr.Column(scale=6):
                gr.Markdown("### 📘 Dictionary Lookup")
                dict_query  = gr.Textbox(placeholder="e.g., Purchasing power")
                dict_output = gr.Textbox(lines=6, label="Definition / Explanation", interactive=False)

                gr.Markdown("### 🔍 Vocabulary Search")
                vocab_input  = gr.Textbox(placeholder="Type substring…")
                vocab_output = gr.Textbox(lines=4, label="Matches", interactive=False)

                gr.Markdown("### 🧠 Analyze Word")
                analyze_input  = gr.Textbox(placeholder="Enter a single word…")
                analyze_output = gr.Textbox(lines=4, label="Analysis", interactive=False)

        # Functional wiring
        init_btn.click(initialize_system, outputs=init_output)
        reset_btn.click(reset_system, outputs=init_output)

        text_input.input(highlight_html, inputs=text_input, outputs=live_preview)

        check_btn.click(split_check, inputs=text_input, outputs=[nonword_output, realword_output])

        auto_btn.click(ui_autocorrect, inputs=text_input, outputs=text_input).then(
            highlight_html, inputs=text_input, outputs=live_preview
        )

        clear_btn.click(clear_text, outputs=text_input).then(
            lambda _: ("", "Ready.", "", ""), outputs=[nonword_output, realword_output, dict_output, vocab_output, analyze_output, init_output]
        ).then(highlight_html, inputs=text_input, outputs=live_preview)

        dict_query.submit(dictionary_lookup, inputs=dict_query, outputs=dict_output)
        vocab_input.submit(search_vocabulary, inputs=vocab_input, outputs=vocab_output)
        analyze_input.submit(analyze_word, inputs=analyze_input, outputs=analyze_output)

    return demo

# 🚀 Step 9: Launch

This final step initializes and launches the Gradio application, allowing users to interact with the spelling correction system in real time.

In [ ]:
# ========== Step 9: Launch ==========
if __name__ == "__main__":
    demo = create_interface()
    demo.queue(max_size=20)
    demo.launch(debug=True, show_error=True, share=True)